<a href="https://colab.research.google.com/github/adtapiagonzalez/PracticasRedesNeuronales_Adrian_Tapia/blob/main/Practica_1_VGG16_en_Keras_Adrian_Tapia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1 . Establecer, cargar y revisar la base de imágenes a utilizar

Se decide cargar los datos de la web de Kaggle

In [ ]:
# Subir el kaggle.json a Colab

from google.colab import files
files.upload()  # Aquí subes la clave kaggle.json

In [ ]:
# Configurar Kaggle dentro de Colab

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Descargar el dataset

!kaggle competitions download -c dogs-vs-cats

In [ ]:
# Descomprimir el ZIP

import zipfile

with zipfile.ZipFile('dogs-vs-cats.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

In [ ]:
# Descomprimir el ZIP de entrenamiento

with zipfile.ZipFile('data/train.zip', 'r') as zip_ref:
    zip_ref.extractall('data/train')

### 2. Separar el conjunto de entrenamiento y el de test

In [ ]:
# Separar el conjunto de entrenamiento y el de test

import os
import shutil
import random

# Ruta en donde están las imágenes
source_data_dir = "data/train/train"

# Nuevas carpetas para el conjunto de datos dividido
base_dir = "data_split"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

# Limpia las carpetas antes de dividir las imágenes
shutil.rmtree(base_dir, ignore_errors=True)

# Se crea la estructura necesaria para ImageDataGenerator:
# data_split/train/cat, data_split/train/dog, data_split/test/cat, data_split/test/dog
for folder_path in ["train/cat", "train/dog", "test/cat", "test/dog"]:
    os.makedirs(os.path.join(base_dir, folder_path), exist_ok=True)

# En este paso, se dividen las imágenes en carpetas distintas en función del nombre de la imagen
all_images = os.listdir(source_data_dir)
cat_images = [img for img in all_images if img.startswith('cat.')]
dog_images = [img for img in all_images if img.startswith('dog.')]

def split_and_copy_class(images_list, class_name, split_ratio=0.8):
    random.shuffle(images_list) # De forma aleatoria
    split_index = int(len(images_list) * split_ratio)

    train_imgs = images_list[:split_index]
    test_imgs = images_list[split_index:]

    # Copia para el train set
    for img_name in train_imgs:
        src = os.path.join(source_data_dir, img_name)
        dst = os.path.join(train_dir, class_name, img_name)
        shutil.copyfile(src, dst)

    # Copia para el test set
    for img_name in test_imgs:
        src = os.path.join(source_data_dir, img_name)
        dst = os.path.join(test_dir, class_name, img_name)
        shutil.copyfile(src, dst)

# Se dividen en un ratio de 0.8 para train
split_and_copy_class(cat_images, "cat", split_ratio=0.8)
split_and_copy_class(dog_images, "dog", split_ratio=0.8)

print("Dataset dividido correctamente")

In [ ]:
# Se importan las librerías necesarias
import keras,os
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPool2D , Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

In [ ]:
# A continuación, se generan las imágenes
trdata = ImageDataGenerator()
traindata = trdata.flow_from_directory(directory="data_split/train",target_size=(224,224)) # Tamaño de 224x224
tsdata = ImageDataGenerator()
testdata = tsdata.flow_from_directory(directory="data_split/test", target_size=(224,224))

### 3. Definir las capas de la red neuronal

In [ ]:
# Arquitectura de la red

model = Sequential() # Se crea un modelo secuencial

# Red VGG16
model.add(Conv2D(input_shape=(224,224,3),filters=64,kernel_size=(3,3),padding="same", activation="relu"))
model.add(Conv2D(filters=64,kernel_size=(3,3),padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))


model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))


model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))


model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))


model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))

In [ ]:
model.add(Flatten())
model.add(Dense(units=4096,activation="relu"))
model.add(Dense(units=4096,activation="relu"))
model.add(Dense(units=2, activation="softmax"))

### 4. Especificar las opciones para el entrenamiento

In [ ]:
from keras.optimizers import Adam
opt = Adam(learning_rate=0.00001) # Optimizador Adam con learning rate de 0.00001
model.compile(optimizer=opt, loss=keras.losses.categorical_crossentropy, metrics=['accuracy']) # loss y accuracy

In [ ]:
# Resumen del modelo
model.summary()

In [ ]:
# Callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping
checkpoint = ModelCheckpoint("vgg16_1.h5",
                             monitor='val_accuracy',
                             verbose=1,
                             save_best_only=True,
                             save_weights_only=False,
                             mode='auto')
early = EarlyStopping(monitor='val_accuracy',
                      min_delta=0,
                      patience=20,
                      verbose=1,
                      mode='auto')

from keras.callbacks import Callback

# Callback para guardar métricas por iteración
class BatchHistory(Callback):
    def on_train_begin(self, logs=None):
        self.batch_acc = []
        self.batch_loss = []

    def on_train_batch_end(self, batch, logs=None):
        self.batch_acc.append(logs.get('accuracy'))
        self.batch_loss.append(logs.get('loss'))

batch_history = BatchHistory()

In [ ]:
# Función  para mostrar el filtro
import matplotlib.pyplot as plt

def plot_filter_weights(layer, filter_index=0):
    filters, biases = layer.get_weights()
    f = filters[:, :, :, filter_index]

    # Normalizar para visualizar
    f_min, f_max = f.min(), f.max()
    f = (f - f_min) / (f_max - f_min)

    plt.imshow(f)
    plt.title(f"Filtro {filter_index}")
    plt.axis('off')
    plt.show()

In [ ]:
# Función para visualizar el mapa de activación
from keras.models import Model
import numpy as np
from tensorflow.keras.preprocessing import image

def get_activation_map(model, layer_name, img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    activation_model = Model(inputs=model.inputs,
                         outputs=model.get_layer(layer_name).output)

    activations = activation_model.predict(img_array)
    return activations

In [ ]:
import os

# Imagen de perro
# Coge una imagen cualquiera de perro del conjunto de evaluación (test)
# Para asegurar que siempre sea la misma imagen, se ordena la lista de archivos.
dog_images_in_test = sorted(os.listdir("data_split/test/dog"))
dog_img_path = "data_split/test/dog/" + dog_images_in_test[0]

In [ ]:
print("La imagen utilizada se encuentra en:", dog_img_path)

from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Cargar imagen
img = image.load_img(dog_img_path, target_size=(224,224))

# Mostrar imagen original
plt.imshow(img)
plt.title("Imagen original (perro)")
plt.axis('off')
plt.show()

In [ ]:
# Primera capa
first_conv_layer = model.layers[0]

# Asegurarse de que el modelo esté construido
# El input_shape debe coincidir con el definido en la primera capa (224,224,3),
# añadiendo None para el tamaño del batch
model.build(input_shape=(None, 224, 224, 3))

# Realizar una inferencia con un input dummy para asegurar que el modelo se 'llama' y el input se define
import numpy as np
dummy_input = np.zeros((1, 224, 224, 3))
_ = model.predict(dummy_input)

# Filtro inicial
plot_filter_weights(first_conv_layer, filter_index=0)

# Guardar los pesos iniciales del primer filtro para comparar más tarde
initial_first_filter_weights_store = first_conv_layer.get_weights()[0][:, :, :, 0]
print(f"Norma de los pesos iniciales del filtro 0: {np.linalg.norm(initial_first_filter_weights_store)}")

# Activación inicial
activations = get_activation_map(model, first_conv_layer.name, dog_img_path)

plt.imshow(activations[0, :, :, 0], cmap='viridis')
plt.title("Mapa activación inicial - primer filtro")
plt.axis('off')
plt.show()

### 5. Entrenar la red con las imágenes de entrenamiento y revisar el resultado

In [ ]:
# Entrenamiento
hist = model.fit(
    x=traindata,
    steps_per_epoch=100,
    epochs=100,
    validation_data=testdata,
    validation_steps=10,
    callbacks=[checkpoint, early, batch_history]
)

In [ ]:
# Evaluación final
loss, acc = model.evaluate(testdata)
print(f"Accuracy final en validación: {acc:.4f}")

#### Gráficas

In [ ]:
# Precisión por iteración en train
import matplotlib.pyplot as plt

plt.figure()
plt.plot(batch_history.batch_acc)
plt.title("Train Accuracy por iteración (batch)")
plt.xlabel("Iteración")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# Loss por iteración en train
plt.figure()
plt.plot(batch_history.batch_loss)
plt.title("Train Loss por iteración (batch)")
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.show()

In [ ]:
# Precisión por época en test
plt.figure()
plt.plot(hist.history['val_accuracy'])
plt.title("Test Accuracy por época")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# Loss por época en test
plt.figure()
plt.plot(hist.history['val_loss'])
plt.title("Test Loss por época")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

#### Filtros y mapas de activación

In [ ]:
# Filtros y mapas de activación
# Filtro aprendido
plot_filter_weights(model.layers[0], filter_index=0)

final_first_filter_weights = model.layers[0].get_weights()[0][:, :, :, 0]
print(f"Norma de los pesos finales del filtro 0: {np.linalg.norm(final_first_filter_weights)}")

# Comparar los pesos iniciales y finales
import numpy as np
if 'initial_first_filter_weights_store' in globals() and not np.allclose(initial_first_filter_weights_store, final_first_filter_weights):
    print("Los pesos del filtro 0 en la primera capa convolucional han cambiado durante el entrenamiento (numéricamente).")
else:
    print("Advertencia: Los pesos del filtro 0 en la primera capa convolucional NO han cambiado significativamente durante el entrenamiento (numéricamente).")

# Activación aprendida
activations = get_activation_map(model, first_conv_layer.name, dog_img_path)

plt.imshow(activations[0, :, :, 0], cmap='viridis')
plt.title("Mapa activación FINAL - primera capa")
plt.axis('off')
plt.show()

In [ ]:
# El acceso al filtro de la última capa se ha visto dificultado al tener alta
# dimensionalidad. Por lo tanto, se modifica el código de la función para mostrar
# el filtro con el objetivo de que se visualice adecuadamente. Se han seleccionado
# los tres primeros canales.

import matplotlib.pyplot as plt
import numpy as np

def plot_filter_weights(layer, filter_index=0):
    filters, biases = layer.get_weights()
    f = filters[:, :, :, filter_index]

    # Normalizar
    f_min, f_max = f.min(), f.max()
    if (f_max - f_min) != 0:
        f = (f - f_min) / (f_max - f_min)

    # Si tiene más de 3 canales, coger los 3 primeros
    if f.shape[-1] >= 3:
        f_display = f[:, :, :3]
    else:
        f_display = f

    plt.imshow(f_display)
    plt.title(f"Filtro {filter_index}")
    plt.axis('off')
    plt.show()

In [ ]:
# Buscar última capa convolucional
last_conv_layer = None
for layer in reversed(model.layers):
    if isinstance(layer, Conv2D):
        last_conv_layer = layer
        break

# Filtro
plot_filter_weights(last_conv_layer, filter_index=0)

# Activación
activations = get_activation_map(model, last_conv_layer.name, dog_img_path)

plt.imshow(activations[0, :, :, 0], cmap='viridis')
plt.title("Mapa activación FINAL - última capa")
plt.axis('off')
plt.show()

Para visualizar la diferencia entre los mapas de activación inicial y final del primer filtro, podemos calcular su resta y mostrarla.

In [ ]:
diff = final_first_filter_weights - initial_first_filter_weights_store

print("Cambio medio:", np.mean(np.abs(diff)))
print("Cambio máximo:", np.max(np.abs(diff)))

#### Sobreajuste

In [ ]:
# Comparación Train vs Test en Precisión
plt.figure()
plt.plot(hist.history['accuracy'], label='Train')
plt.plot(hist.history['val_accuracy'], label='Test')
plt.legend()
plt.title("Accuracy: Train vs Test")
plt.show()

In [ ]:
# Comparación Train vs Test en Loss
plt.figure()
plt.plot(hist.history['loss'], label='Train')
plt.plot(hist.history['val_loss'], label='Test')
plt.legend()
plt.title("Loss: Train vs Test")
plt.show()

In [ ]:
# Cumplimiento del supuesto de Normalidad y Homocedasticidad

from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

# Obtener las precisiones de las últimas 5 épocas
train_acc_last_5 = hist.history['accuracy'][-5:]
test_acc_last_5 = hist.history['val_accuracy'][-5:]

# Calcular las diferencias entre las precisiones (asunción de normalidad es sobre las diferencias)
differences = np.array(train_acc_last_5) - np.array(test_acc_last_5)

# --- 1. Prueba de Shapiro-Wilk para normalidad ---
# Se utiliza para comprobar si una muestra procede de una distribución normal.
# Hipótesis nula (H0): La muestra procede de una distribución normal.
# Hipótesis alternativa (H1): La muestra no procede de una distribución normal.

print("\n--- Verificación de Normalidad (Shapiro-Wilk) ---")
if len(differences) >= 3: # Shapiro-Wilk requiere al menos 3 muestras
    stat, p = stats.shapiro(differences)
    print(f'Estadístico de Shapiro-Wilk: {stat:.4f}, p-valor: {p:.4f}')

    alpha = 0.05
    if p > alpha:
        print('Las diferencias parecen seguir una distribución normal (no se puede rechazar H0)')
    else:
        print('Las diferencias no parecen seguir una distribución normal (se rechaza H0)')
else:
    print('Se necesitan al menos 3 épocas para realizar la prueba de Shapiro-Wilk.')

# --- 2. Prueba de Levene para Homogeneidad de Varianzas ---
# Se utiliza para comprobar si dos o más muestras tienen varianzas iguales.
# Hipótesis nula (H0): Las varianzas son iguales.
# Hipótesis alternativa (H1): Al menos una varianza es diferente.

print("\n--- Verificación de Homogeneidad de Varianzas (Levene's Test) ---")
if len(train_acc_last_5) >= 2 and len(test_acc_last_5) >= 2: # Levene's test requiere al menos 2 muestras por grupo
    stat_levene, p_levene = stats.levene(train_acc_last_5, test_acc_last_5)
    print(f'Estadístico de Levene: {stat_levene:.4f}, p-valor: {p_levene:.4f}')

    alpha = 0.05
    if p_levene > alpha:
        print('Las varianzas de las precisiones de train y test son homogéneas (no se puede rechazar H0)')
    else:
        print('Las varianzas de las precisiones de train y test NO son homogéneas (se rechaza H0)')
else:
    print('Se necesitan al menos 2 épocas en cada conjunto para realizar la prueba de Levene.')


# --- 3. Visualización de la distribución de las diferencias ---

plt.figure(figsize=(12, 5))

# Histograma de las diferencias
plt.subplot(1, 2, 1)
plt.hist(differences, bins=2, edgecolor='black') # Ajustar bins si hay más datos
plt.title('Histograma de las Diferencias de Precisión')
plt.xlabel('Diferencia (Train Acc - Test Acc)')
plt.ylabel('Frecuencia')

# Gráfico Q-Q (Quantile-Quantile Plot)
plt.subplot(1, 2, 2)
sm.qqplot(differences, line='s', ax=plt.gca())
plt.title('Q-Q Plot de las Diferencias de Precisión')

plt.tight_layout()
plt.show()

In [ ]:
# Prueba t
from scipy.stats import ttest_rel

# Tomar las precisiones de las últimas 5 épocas
train_acc_last_5 = hist.history['accuracy'][-5:]
test_acc_last_5 = hist.history['val_accuracy'][-5:]

# Asegurarse de que haya al menos 5 épocas para la prueba
if len(train_acc_last_5) < 5 or len(test_acc_last_5) < 5:
    print("No hay suficientes épocas (menos de 5) para realizar la prueba t en las últimas 5 épocas.")
else:
    t_stat, p_value = ttest_rel(train_acc_last_5, test_acc_last_5)

    print(f"T-test (últimas 5 épocas): t={t_stat:.4f}, p={p_value:.4f}")


In [ ]:
# ¿Cuándo empieza el sobreajuste?
import numpy as np
from scipy.stats import ttest_rel

train_acc = hist.history['accuracy']
test_acc = hist.history['val_accuracy']
alpha = 0.05

overfit_epoch = -1 # Inicializar a -1 para indicar que no se detectó sobreajuste aún

# ttest_rel requiere al menos 2 muestras para calcular la varianza.
# Por lo tanto, debemos comenzar a verificar desde la segunda época (índice 1),
# usando los datos hasta esa época (por ejemplo, [acc_0, acc_1]).
min_epochs_for_test = 2

# Iterar a través de las épocas, comenzando desde el número mínimo de épocas requerido para una prueba t.
for i in range(min_epochs_for_test - 1, len(train_acc)): # 'i' es el índice de la época (base 0)
    # Tomar las precisiones desde la época 0 hasta la época actual 'i' (inclusive)
    current_train_samples = train_acc[:i+1]
    current_test_samples = test_acc[:i+1]

    # Realizar una prueba t pareada con la hipótesis alternativa de que train_acc > test_acc
    # Estamos interesados en una prueba unilateral donde la precisión de entrenamiento es significativamente mayor que la precisión de prueba.
    t_stat, p_value = ttest_rel(current_train_samples, current_test_samples, alternative='greater')

    if p_value < alpha:
        overfit_epoch = i # Almacenar el índice de la época (base 0)
        break

if overfit_epoch != -1:
    # Se suma 1 para convertir de un índice de época (base 0) a un número de época legible para humanos (base 1)
    print(f"Sobreajuste estadísticamente significativo (con alfa={alpha}) empieza aproximadamente en época: {overfit_epoch + 1}")
else:
    print(f"No se detectó sobreajuste estadísticamente significativo (con alfa={alpha}) en el transcurso del entrenamiento.")
